In [2]:
import os
import requests
from pathlib import Path
from Bio import SeqIO

In [3]:
os.chdir("/Users/jairtorres/Desktop/HLA_DQ_modeling")

In [4]:
ALLELES = [
    # DQA1
    "DQA1*05:01:01:01",
    "DQA1*05:05:01:01",
    "DQA1*02:01:01:01",
    "DQA1*03:01:01:01",
    # DQB1
    "DQB1*02:01:01:01",
    "DQB1*02:02:01:01",
    "DQB1*03:01:01:01",
    "DQB1*03:02:01:01",
]

UNIPROT_REFERENCE = {
    "DQA1": "P01903",
    "DQB1": "P05538",
}

RAW_FOLDER     = Path("fastas_raw")
TRIMMED_FOLDER = Path("fastas_trimmed")

In [5]:
def fetch_imgt_allele(allele_name, output_folder):
    if not allele_name.startswith("HLA-"):
        allele_name_full = f"HLA-{allele_name}"
    else:
        allele_name_full = allele_name

    url = f"https://www.ebi.ac.uk/cgi-bin/ipd/api/allele/{allele_name_full}"
    r = requests.get(url)
    r.raise_for_status()
    data = r.json()

    sequence = data["sequence"]["protein"]["sequence"]
    allele_id = data["name"]

    safe_name = allele_name.replace("*", "_").replace(":", "_")
    out_path = output_folder / f"{safe_name}.fasta"

    with open(out_path, "w") as f:
        f.write(f">{allele_id}\n{sequence}\n")

    print(f"  [OK] {allele_id}: {len(sequence)} aa -> {out_path.name}")
    return out_path

In [6]:
def get_uniprot_features(uniprot_id):
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
    r = requests.get(url)
    r.raise_for_status()
    data = r.json()

    features = {"signal_peptide": None, "transmembrane": None}

    for feature in data.get("features", []):
        ftype = feature.get("type", "")
        start = feature["location"]["start"]["value"]
        end   = feature["location"]["end"]["value"]

        if ftype == "Signal" and features["signal_peptide"] is None:
            features["signal_peptide"] = (start, end)
        elif ftype == "Transmembrane" and features["transmembrane"] is None:
            features["transmembrane"] = (start, end)

    return features

In [7]:
def extract_extracellular_domain(sequence, features):
    start = features["signal_peptide"][1] if features["signal_peptide"] else 0
    end   = features["transmembrane"][0] - 1 if features["transmembrane"] else len(sequence)
    return sequence[start:end]

def trim_allele_fasta(fasta_path, features, output_folder):
    records = list(SeqIO.parse(fasta_path, "fasta"))
    out_lines = []

    for record in records:
        seq = str(record.seq)
        trimmed = extract_extracellular_domain(seq, features)
        print(f"  [TRIM] {record.id}: {len(seq)} aa -> {len(trimmed)} aa")
        out_lines.append(f">{record.id}_extracellular\n{trimmed}")

    out_path = output_folder / fasta_path.name.replace(".fasta", "_extracellular.fasta")
    with open(out_path, "w") as f:
        f.write("\n".join(out_lines) + "\n")

    return out_path

In [8]:
RAW_FOLDER.mkdir(exist_ok=True)
TRIMMED_FOLDER.mkdir(exist_ok=True)

In [9]:
print("\n=== Fetching UniProt boundary annotations ===")
boundaries = {}
for chain, uniprot_id in UNIPROT_REFERENCE.items():
    features = get_uniprot_features(uniprot_id)
    boundaries[chain] = features
    print(f"  {chain} ({uniprot_id})")
    print(f"    Signal peptide: {features['signal_peptide']}")
    print(f"    Transmembrane:  {features['transmembrane']}")


=== Fetching UniProt boundary annotations ===
  DQA1 (P01903)
    Signal peptide: (1, 25)
    Transmembrane:  (217, 239)
  DQB1 (P05538)
    Signal peptide: (1, 32)
    Transmembrane:  (230, 250)


In [1]:
for k, v in trimmed_seqs.items():
    print(f"{k}: {v[:6]}...{v[-6:]}")

NameError: name 'trimmed_seqs' is not defined

In [15]:
from Bio import SeqIO
from io import StringIO

print("Downloading full HLA protein FASTA from IMGT/HLA GitHub...")
url = "https://raw.githubusercontent.com/ANHIG/IMGTHLA/Latest/hla_prot.fasta"
r = requests.get(url)
r.raise_for_status()
hla_prot_fasta = r.text
print(f"  Downloaded {len(hla_prot_fasta):,} characters")

hla_db = {}
for record in SeqIO.parse(StringIO(hla_prot_fasta), "fasta"):

    parts = record.description.split()
    if len(parts) >= 2:
        allele_name = parts[1]  
        hla_db[allele_name] = str(record.seq)

print(f"  Parsed {len(hla_db):,} alleles")

print("\n=== Fetching allele sequences from IMGT/HLA ===")
fetched = []
for allele in ALLELES:
    if allele in hla_db:
        sequence = hla_db[allele]
        safe_name = allele.replace("*", "_").replace(":", "_")
        out_path = RAW_FOLDER / f"{safe_name}.fasta"
        with open(out_path, "w") as f:
            f.write(f">{allele}\n{sequence}\n")
        print(f"  [OK] {allele}: {len(sequence)} aa -> {out_path.name}")
        fetched.append((allele, out_path))
    else:
        print(f"  [ERROR] {allele}: not found in database")

  Downloaded 14,464,279 characters
  Parsed 45,762 alleles

=== Fetching allele sequences from IMGT/HLA ===
  [OK] DQA1*05:01:01:01: 254 aa -> DQA1_05_01_01_01.fasta
  [OK] DQA1*05:05:01:01: 254 aa -> DQA1_05_05_01_01.fasta
  [OK] DQA1*02:01:01:01: 254 aa -> DQA1_02_01_01_01.fasta
  [OK] DQA1*03:01:01:01: 255 aa -> DQA1_03_01_01_01.fasta
  [OK] DQB1*02:01:01:01: 261 aa -> DQB1_02_01_01_01.fasta
  [OK] DQB1*02:02:01:01: 261 aa -> DQB1_02_02_01_01.fasta
  [OK] DQB1*03:01:01:01: 261 aa -> DQB1_03_01_01_01.fasta
  [OK] DQB1*03:02:01:01: 261 aa -> DQB1_03_02_01_01.fasta


In [16]:
print("\n=== Trimming to extracellular domain ===")
for allele, fasta_path in fetched:
    chain = "DQA1" if "DQA1" in allele else "DQB1"
    features = boundaries[chain]
    try:
        trim_allele_fasta(fasta_path, features, TRIMMED_FOLDER)
    except Exception as e:
        print(f"  [ERROR] trimming {allele}: {e}")

print(f"\n=== Done! Trimmed FASTAs saved to '{TRIMMED_FOLDER}/' ===")


=== Trimming to extracellular domain ===
  [TRIM] DQA1*05:01:01:01: 254 aa -> 191 aa
  [TRIM] DQA1*05:05:01:01: 254 aa -> 191 aa
  [TRIM] DQA1*02:01:01:01: 254 aa -> 191 aa
  [TRIM] DQA1*03:01:01:01: 255 aa -> 191 aa
  [TRIM] DQB1*02:01:01:01: 261 aa -> 197 aa
  [TRIM] DQB1*02:02:01:01: 261 aa -> 197 aa
  [TRIM] DQB1*03:01:01:01: 261 aa -> 197 aa
  [TRIM] DQB1*03:02:01:01: 261 aa -> 197 aa

=== Done! Trimmed FASTAs saved to 'fastas_trimmed/' ===


In [5]:
PAIRS = [
    ("DQA1*05:01:01:01", "DQB1*02:01:01:01"),  # DQ2.5 cis - high risk celiac
    ("DQA1*05:05:01:01", "DQB1*02:02:01:01"),  # DQ2.5 trans - high risk celiac
    ("DQA1*02:01:01:01", "DQB1*02:02:01:01"),  # DQ2.2 - low risk
    ("DQA1*05:05:01:01", "DQB1*03:01:01:01"),  # DQ7.5 - very low risk
    ("DQA1*03:01:01:01", "DQB1*03:02:01:01"),  # DQ8 - low risk
]

PAIRS_FOLDER = Path("alphafold_inputs")
PAIRS_FOLDER.mkdir(exist_ok=True)

trimmed_seqs = {}
for fasta_path in TRIMMED_FOLDER.glob("*.fasta"):
    records = list(SeqIO.parse(fasta_path, "fasta"))
    if records:
        allele = records[0].id.replace("_extracellular", "")
        trimmed_seqs[allele] = str(records[0].seq)

print("Loaded trimmed sequences:")
for k, v in trimmed_seqs.items():
    print(f"  {k}: {len(v)} aa")

print("\n=== Generating AlphaFold-Multimer inputs ===")
for alpha, beta in PAIRS:
    if alpha not in trimmed_seqs:
        print(f"  [SKIP] {alpha} not found")
        continue
    if beta not in trimmed_seqs:
        print(f"  [SKIP] {beta} not found")
        continue

    combined = f"{trimmed_seqs[alpha]}:{trimmed_seqs[beta]}"

    safe_alpha = alpha.replace("*", "_").replace(":", "_")
    safe_beta  = beta.replace("*", "_").replace(":", "_")
    fname = f"{safe_alpha}__{safe_beta}.fasta"
    out_path = PAIRS_FOLDER / fname

    with open(out_path, "w") as f:
        f.write(f">{alpha}__{beta}\n{combined}\n")

    print(f"  [OK] {fname}")

print(f"\n=== AlphaFold inputs saved to '{PAIRS_FOLDER}/' ===")

Loaded trimmed sequences:
  DQA1*05:05:01:01: 191 aa
  DQA1*02:01:01:01: 191 aa
  DQB1*03:02:01:01: 197 aa
  DQB1*02:01:01:01: 197 aa
  DQA1*05:01:01:01: 191 aa
  DQB1*03:01:01:01: 197 aa
  DQB1*02:02:01:01: 197 aa
  DQA1*03:01:01:01: 191 aa

=== Generating AlphaFold-Multimer inputs ===
  [OK] DQA1_05_01_01_01__DQB1_02_01_01_01.fasta
  [OK] DQA1_05_05_01_01__DQB1_02_02_01_01.fasta
  [OK] DQA1_02_01_01_01__DQB1_02_02_01_01.fasta
  [OK] DQA1_05_05_01_01__DQB1_03_01_01_01.fasta
  [OK] DQA1_03_01_01_01__DQB1_03_02_01_01.fasta

=== AlphaFold inputs saved to 'alphafold_inputs/' ===
